[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/JoneSu1/DeepCompare_batch_mark/blob/main/notebooks/run_colab.ipynb)

# MPRA benchmark on Colab — DeepCompARE vs Enformer vs AlphaGenome

1. **Runtime → Change runtime type → T4 GPU** (Enformer needs it; DeepCompARE runs on CPU).
2. Edit `REPO_URL` in the first cell to your GitHub repo (HTTPS, public or with token).
3. Run all cells. AlphaGenome asks for an API key via the form (free, non-commercial:
   https://deepmind.google.com/science/alphagenome).

In [ ]:
REPO_URL = "https://github.com/JoneSu1/DeepCompare_batch_mark"
!git clone -q {REPO_URL} mpra-benchmark
%cd mpra-benchmark

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-deps "git+https://github.com/anderssonlab/deepISA"
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('
' + '!' * 70)
    print('NO GPU DETECTED. Enformer will refuse to run on CPU (~34 h).')
    print('Fix now: 代码执行程序/Runtime -> 更改运行时类型/Change runtime type -> T4 GPU -> 保存/Save')
    print('then Run all again.')
    print('!' * 70 + '
')


## Step 1 — sequences (downloads the 6 needed hg38 chromosomes from UCSC, ~5 min)

Always run this for the **full** benchmark: the repo ships MVP-only artifacts
(HBB+HBG1, 606 variants) and this step overwrites them with the full 3,877-variant set.

In [ ]:
import os
os.makedirs('data/reference', exist_ok=True)
for c in [1, 2, 11, 19, 22, 'X']:
    path = f'data/reference/chr{c}.fa'
    if not os.path.exists(path):
        !curl -sL "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr{c}.fa.gz" | gunzip > {path}
    print(path, 'ready')
!python scripts/01_prepare_sequences.py --fasta data/reference/chr1.fa data/reference/chr2.fa data/reference/chr11.fa data/reference/chr19.fa data/reference/chr22.fa data/reference/chrX.fa

In [ ]:
!python scripts/02_predict_deepcompare.py --device cpu

In [ ]:
# Enformer on T4: ~3,885 contexts (3,877 mut + 8 unique refs), expect ~15-30 min total
# T4 has 16 GB VRAM; if you hit a CUDA OOM, rerun with --batch-size 4
!python scripts/03_predict_enformer.py --device auto --batch-size 8


## Step 4 — AlphaGenome (API; ~3,885 calls, resumable via SQLite cache)

In [ ]:
# @title Fill in your AlphaGenome API key
API_KEY = ""  # @param {type:"string"}
assert API_KEY, 'paste your key from https://deepmind.google.com/science/alphagenome'
with open('configs/alphagenome.yaml', 'w') as f:
    f.write(open('configs/alphagenome.yaml.example').read().replace('${ALPHAGENOME_API_KEY}', API_KEY))
!python scripts/04_predict_alphagenome.py

In [ ]:
!python scripts/05_correlations.py

In [ ]:
import os
import pandas as pd
corr = pd.read_csv('results/correlations.csv')
pivot = corr.pivot_table(index='Element', columns=['Method', 'Track'], values='Correlation')
display(pivot.round(2))
from google.colab import files
for f in ['results/correlations.csv',
          'results/predictions/deepcompare.csv',
          'results/predictions/enformer.csv',
          'results/predictions/alphagenome.csv']:
    if os.path.exists(f):
        files.download(f)
    else:
        print(f'skipped (not produced): {f}')